### Импорт библиотек

In [1]:
import pandas as pd
import psycopg2
import io, csv
from datetime import datetime, timedelta

### Подключения к БД

In [2]:
host = 'rc1a-b58fi5jrta1cj7v5.mdb.yandexcloud.net'
db = 'teacher'
login = 'login'
password = 'password'

def pg_connect(host, db, login, password):
    return psycopg2.connect(database=db, user=login, password=password, port=6432, host=host)

### Получаем список проверок

In [3]:
columns = ['check_id', 'subject', 'name', 'description', 'sql_function']

with pg_connect(host, db, login, password) as con_gp:
    con_gp.autocommit = True
    gp_query = f"""select * from test_sch.checks"""
    
    with con_gp.cursor() as cursor:
        cursor.execute(gp_query)
        checks = pd.DataFrame(data=cursor.fetchall(), columns=columns)

pd.set_option('display.max_colwidth', None)
display(checks.T)

,0
check_id,1
subject,transactions
name,trn_null_cus
description,Проверка на наличие строк с пустым значением в поле customer_id
sql_function,test_sch.dq_check_trn_null_customer


### Выполняем проверки

In [7]:
columns = ['is_productive', 'dt', 'check_id', 'check_done', 'problem_count', 'problem_damage', 'error_description']
is_productive = False
error_desc = None
check_row = pd.DataFrame(data=[[None for i in range(len(columns))]], columns=columns)

for check in checks.itertuples(index=False):

    dt = "'2025-04-02'"

    try:
    
        with pg_connect(host, db, login, password) as con_gp:
            con_gp.autocommit = True
            gp_query = f"""select * from {check.sql_function}({is_productive}, {dt}) as (problem_count integer, problem_damage float)"""
            print(gp_query)
            
            with con_gp.cursor() as cursor:
                cursor.execute(gp_query)
                df = pd.DataFrame(data=cursor.fetchall(), columns=['problem_count', 'problem_damage'])

        check_row['problem_count'] = df['problem_count']
        check_row['problem_damage'] = df['problem_damage']
        check_row['check_done'] = df['problem_count'].apply(lambda x: False if x > 0 else True)

    except psycopg2.Error as e:
        error_desc = e.pgerror.split('\n')[0]
        check_row['error_description'] = error_desc
    finally:
        check_row['is_productive'] = is_productive
        check_row['dt'] = dt
        check_row['check_id'] = check.check_id

    display(check_row)

select * from test_sch.dq_check_trn_null_customer(False, '2025-04-02') as (problem_count integer, problem_damage float)


,is_productive,dt,check_id,check_done,problem_count,problem_damage,error_description
0,False,'2025-04-02',1,True,0,None,None


### Записываем лог проверки

In [8]:
with pg_connect(host, db, login, password) as con_gp:

    for index, row in check_row.iterrows():
        
        gp_query = f"""
            insert into test_sch.checks_history
            values(
                nextval('test_sch.checks_history_id_seq'), %s, %s, %s, %s, %s, %s, %s
            );
        """

        try:
            with con_gp.cursor() as cursor:
                cursor.execute(
                    gp_query,
                    (
                        row['is_productive'],
                        row['dt'],
                        row['check_id'],
                        row['check_done'],
                        row['problem_count'],
                        row['problem_damage'],
                        row['error_description']
                    )
                )
                con_gp.commit()
                print(f"Лог проверки {row['check_id']} за {row['dt']} успешно записан")
        except psycopg2.Error as e:
            con_gp.rollback()
            print(f"Ошибка записи лога проверки {row['check_id']} за {row['dt']}: {e}")

Лог проверки 1 за '2025-04-02' успешно записан
